# 🎨 The Policy Briefing in 40 Minutes
### Unit 1 Activity — Colour Theory in Data Visualization (UE24CS342AA9)

---

**11:00 AM.** You're a junior analyst at **GlobalIndex Insights**. In 40 minutes you're
presenting well-being trends to a policy think-tank. Your Creative Director, Meera,
left these notes:

> *"Every chart in this draft misuses color somewhere -- wrong scale type, no
> accessible palette, a rainbow colormap, you name it. Fix each one using the right
> color-mapping principle. I've marked what's broken with a `# TODO`."*

Same dataset the team has been using: the World Happiness Report 2015.

## ⏱️ Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import TwoSlopeNorm
import time

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

df = pd.read_csv("whr2015.csv")
mission_start = time.time()
print(f"Loaded {len(df)} rows. 40 minutes on the clock. Go.")
df.head()

In [ ]:
# Run any time to check your remaining budget
elapsed_min = (time.time() - mission_start) / 60
remaining = max(0, 40 - elapsed_min)
print(f"Elapsed: {elapsed_min:0.1f} min   |   Remaining: {remaining:0.1f} min")

---
## Task 1 — Pick the Right Aesthetic *(~4 min)*

Meera's note: *"We need to compare Economy (GDP per Capita) across the top 10
countries. Right now it's plotted with only position -- add color as a SECOND aesthetic
encoding the same variable, so the redundancy makes the comparison unmistakable."*

In [ ]:
top10 = df.nsmallest(10, "Happiness Rank")

fig, ax = plt.subplots(figsize=(7, 5))

# TODO 1: add a 'color=' argument to barh that maps to top10["Economy (GDP per Capita)"]
# using a sequential colormap (e.g. cmap via a Normalize + plt.cm, or just pass a cmap
# array). Hint: you can compute colors with plt.cm.Blues(plt.Normalize()(values)).
values = top10["Economy (GDP per Capita)"]
bar_colors = ...

ax.barh(top10["Country"], values, color=bar_colors)
ax.set_title("Top 10 Countries: Economy (GDP per Capita)")
plt.tight_layout()
plt.show()

---
## Task 2 — Build the Region x Factor Heatmap *(~5 min)*

Meera's note: *"I want the exact 'average value in a colored matrix' chart from the
lecture -- rows are regions, columns are well-being factors, cell color is the average
score. Use a sequential colormap since there's no natural zero-midpoint here."*

In [ ]:
factor_cols = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
               "Freedom", "Trust (Government Corruption)", "Generosity"]

# TODO 2: build the region x factor matrix (mean of each factor, grouped by Region)
region_matrix = ...

fig, ax = plt.subplots(figsize=(9, 5.5))

# TODO 2b: choose an appropriate SEQUENTIAL colormap name (not "jet"/"rainbow")
im = ax.imshow(region_matrix.values, cmap=..., aspect="auto")

ax.set_xticks(range(len(factor_cols)))
ax.set_xticklabels([c.split(" (")[0] for c in factor_cols], rotation=30, ha="right")
ax.set_yticks(range(len(region_matrix)))
ax.set_yticklabels(region_matrix.index)
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Average Well-Being Factor Scores by Region")
plt.tight_layout()
plt.show()

---
## Task 3 — Fix the Qualitative Scale *(~5 min)*

Meera's note: *"This regional comparison chart currently colors every country the same
color -- there's no way to see the regional pattern. Give each region its own
consistent, equally-weighted color."*

In [ ]:
ordered = df.sort_values("Happiness Score", ascending=True)
regions = ordered["Region"].unique()

# TODO 3: build a color_map dict {region_name: color} giving each region a distinct,
# equally-weighted color (hint: plt.cm.tab10 or plt.cm.Set2 evenly spaced).
color_map = ...

bar_colors = ordered["Region"].map(color_map)

fig, ax = plt.subplots(figsize=(7, 11))
ax.barh(ordered["Country"], ordered["Happiness Score"], color=bar_colors)
ax.tick_params(axis="y", labelsize=5)
ax.set_title("All Countries Colored by Region")

handles = [patches.Patch(color=color_map[r], label=r) for r in regions]
ax.legend(handles=handles, loc="lower right", fontsize=7)
plt.tight_layout()
plt.show()

---
## Task 4 — Build a Diverging Scale *(~5 min)*

Meera's note: *"I want Freedom scores shown as deviation from the GLOBAL AVERAGE, not
raw values -- above-average countries should read as one color family, below-average as
another, meeting at a neutral midpoint. Use a diverging colormap centered at zero."*

In [ ]:
# TODO 4a: compute each country's Freedom score minus the global mean Freedom score
freedom_dev = ...

df_dev = df.assign(freedom_dev=freedom_dev)
sample = pd.concat([df_dev.nlargest(10, "freedom_dev"), df_dev.nsmallest(10, "freedom_dev")])
sample = sample.sort_values("freedom_dev")

# TODO 4b: create a TwoSlopeNorm centered at 0 using sample["freedom_dev"]'s min/max
norm = ...

fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(sample["Country"], sample["freedom_dev"], color=plt.cm.RdBu_r(norm(sample["freedom_dev"])))
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Freedom Score vs. Global Average (10 highest & lowest)")
plt.tight_layout()
plt.show()

---
## Task 5 — Build an Accent Scale *(~6 min)*

Meera's note: *"Within Southern Asia, I want ONE chart that mutes every country to gray
EXCEPT the country with the highest Generosity score and the country with the lowest --
those two are the story. Use vivid, contrasting accent colors for just those two."*

In [ ]:
region_df = df[df["Region"] == "Southern Asia"].sort_values("Generosity")

# TODO 5: find the country name with the LOWEST Generosity and the HIGHEST Generosity
# in region_df (two strings).
lowest_country = ...
highest_country = ...

colors = ["#d9d9d9"] * len(region_df)
colors[list(region_df["Country"]).index(lowest_country)] = "#D55E00"
colors[list(region_df["Country"]).index(highest_country)] = "#0072B2"

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(region_df["Country"], region_df["Generosity"], color=colors)
ax.set_title("Southern Asia: Generosity Score\n(muted baseline + two accent outliers)")
plt.tight_layout()
plt.show()

---
## Task 6 — Kill the Rainbow Colormap *(~5 min)*

Meera's note: *"Whoever built this heatmap used 'jet'. Convert it to grayscale first so
you can SEE why that's a problem, then replace it with a perceptually uniform
colormap."*

In [ ]:
matrix = df.groupby("Region")[factor_cols].mean().values

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].imshow(matrix, cmap="jet", aspect="auto")
axes[0].set_title("Current: 'jet' colormap", fontsize=10)

# TODO 6a: convert the 'jet'-mapped matrix to grayscale using luminance weights
# (0.2126, 0.7152, 0.0722) applied to the RGB channels from plt.cm.jet(...)
gray_jet = ...
axes[1].imshow(gray_jet, cmap="gray", aspect="auto")
axes[1].set_title("'jet' converted to grayscale\n(is the lightness monotonic?)", fontsize=10)

# TODO 6b: re-plot the SAME matrix with a perceptually uniform colormap instead of 'jet'
axes[2].imshow(matrix, cmap=..., aspect="auto")
axes[2].set_title("Fixed: perceptually uniform colormap", fontsize=10)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

---
## Reflection *(~5 min)*

Answer briefly in this markdown cell:

1. Which of the six fixes above would most change a policymaker's takeaway if left
   broken, and why?
2. Explain in your own words why the diverging scale (Task 4) needed a *centered*
   normalization instead of just min-to-max scaling.
3. Why is "it just needs to look distinct" not a sufficient bar for a qualitative color
   scale (Task 3)? What's the extra requirement from the lecture?
4. Complete the lecture's core idea in one sentence: a good color scale should
   *__________* the viewer, not *__________* them.

**Your answers:**

*Type your answers here.*